# 📩 Detección de spam en SMS con RNN / LSTM / GRU

**Objetivo:** clasificar mensajes SMS como *spam* o *ham* usando redes recurrentes en TensorFlow/Keras.

**Contenido:** 1) Dataset · 2) EDA · 3) Limpieza · 4) División train/val/test · 5) Vectorización ·
6) Modelos y experimentos · 7) Gráficas · 8) Evaluación en test · 9) Predicciones · 10) Guardado del modelo.

> 💡 Activa GPU: *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU* (opcional, el dataset es pequeño).

## 0. Configuración

In [ ]:
import os, io, re, json, random, zipfile, urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
sns.set_theme(style="whitegrid")

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 1. Selección y descripción del dataset

**SMS Spam Collection** (UCI): 5.574 mensajes SMS en inglés etiquetados como `ham` (legítimo) o `spam`.

- **Por qué es secuencial:** el significado y la intención dependen del orden de las palabras y su contexto.
- **Por qué es interesante:** las clases están desbalanceadas (~13 % spam), por lo que la *accuracy* sola no basta;
  usaremos precisión, recall, F1 y ROC-AUC.
- **Fuente:** https://archive.ics.uci.edu/dataset/228/sms+spam+collection

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/SMSSpamCollection"):
    with urllib.request.urlopen(URL) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extract("SMSSpamCollection", "data")

df = pd.read_csv("data/SMSSpamCollection", sep="\t", header=None, names=["label", "text"])
print("Forma:", df.shape)
df.head(10)

## 2. Análisis exploratorio de datos (EDA)

In [ ]:
print("Nulos por columna:\n", df.isna().sum(), "\n")
print("Duplicados:", df.duplicated(subset="text").sum())
print("\nDistribución de clases:")
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True).round(3))

fig, ax = plt.subplots(figsize=(4.5, 3.5))
sns.countplot(data=df, x="label", ax=ax, palette=["#4c72b0", "#dd8452"], hue="label", legend=False)
ax.set_title("Distribución de clases"); plt.show()

In [ ]:
df["n_palabras"] = df["text"].str.split().str.len()
df["n_caracteres"] = df["text"].str.len()
print(df.groupby("label")[["n_palabras", "n_caracteres"]].describe().T.round(1))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
sns.histplot(data=df, x="n_palabras", hue="label", bins=40, ax=axes[0], element="step")
axes[0].set_title("Palabras por mensaje")
sns.boxplot(data=df, x="label", y="n_caracteres", ax=axes[1], hue="label", legend=False)
axes[1].set_title("Caracteres por mensaje")
plt.tight_layout(); plt.show()
print("Percentil 95 de palabras:", int(np.percentile(df["n_palabras"], 95)))

In [ ]:
def top_words(texts, n=15):
    c = Counter()
    for t in texts:
        c.update(w for w in re.findall(r"[a-z']+", t.lower()) if w not in ENGLISH_STOP_WORDS and len(w) > 2)
    return c.most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, lab, col in zip(axes, ["ham", "spam"], ["#4c72b0", "#dd8452"]):
    words, counts = zip(*top_words(df.loc[df.label == lab, "text"]))
    ax.barh(words[::-1], counts[::-1], color=col)
    ax.set_title(f"Palabras más frecuentes: {lab}")
plt.tight_layout(); plt.show()

**Observaciones esperadas:** el spam es en promedio más largo, usa palabras como *free, call, txt, claim, prize*,
y contiene números y símbolos de moneda. Por eso en la limpieza **no** borraremos números ni símbolos a ciegas,
sino que los normalizaremos a tokens.

## 3. Limpieza y preprocesamiento

- Eliminar duplicados exactos.
- Minúsculas.
- Normalizar URLs, emails, números largos (teléfonos/códigos) y símbolos de moneda a tokens especiales.
- La puntuación restante la elimina la capa `TextVectorization` dentro del modelo.

> ⚠️ Esta misma función `clean_text` está en `utils.py` y se usa en `predict.py`/`app.py`
> para que el preprocesamiento en inferencia sea idéntico.

In [ ]:
URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
LONGNUM_RE = re.compile(r"\b\d{5,}\b")
MONEY_RE = re.compile(r"[£$€]")

def clean_text(t):
    t = str(t).lower()
    t = URL_RE.sub(" urltoken ", t)
    t = EMAIL_RE.sub(" emailtoken ", t)
    t = LONGNUM_RE.sub(" numtoken ", t)
    t = MONEY_RE.sub(" moneytoken ", t)
    return re.sub(r"\s+", " ", t).strip()

df = df.drop_duplicates(subset="text").dropna().reset_index(drop=True)
df["y"] = (df["label"] == "spam").astype("int32")
df["clean"] = df["text"].apply(clean_text)
print("Tras eliminar duplicados:", df.shape)
df[["text", "clean"]].sample(5, random_state=SEED).style.set_properties(**{"text-align": "left"})

## 4. División en entrenamiento, validación y prueba

70 % / 15 % / 15 %, **estratificada** para conservar la proporción de spam.
Se divide **antes** de construir el vocabulario para evitar fuga de datos.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["y"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["y"], random_state=SEED)

for nombre, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"{nombre:5}: {len(d):5} mensajes | % spam = {d['y'].mean():.3f}")

## 5. Vectorización

`TextVectorization` convierte texto a secuencias de enteros. Se **adapta solo con entrenamiento**.
`max_len` = percentil 95 de longitudes (no el máximo, para no rellenar de más).
Al incluirla **dentro del modelo**, el modelo guardado recibe texto crudo.

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = int(np.percentile([len(t.split()) for t in train_df["clean"]], 95))
print("MAX_LEN =", MAX_LEN)

vectorizer = layers.TextVectorization(max_tokens=VOCAB_SIZE, output_mode="int",
                                      output_sequence_length=MAX_LEN,
                                      standardize="lower_and_strip_punctuation")
vectorizer.adapt(np.array(train_df["clean"], dtype=str))
print("Tamaño de vocabulario:", vectorizer.vocabulary_size())
print("Primeras palabras:", vectorizer.get_vocabulary()[:15])

ejemplo = train_df["clean"].iloc[0]
print("\nTexto :", ejemplo)
print("Tokens:", vectorizer(tf.constant([ejemplo]))[0].numpy())

BATCH = 32
def make_ds(X, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((np.array(X, dtype=str), np.array(y, dtype="float32")))
    if shuffle:
        ds = ds.shuffle(len(X), seed=SEED)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_df["clean"], train_df["y"], shuffle=True)
val_ds = make_ds(val_df["clean"], val_df["y"])
test_ds = make_ds(test_df["clean"], test_df["y"])

## 6. Construcción y entrenamiento de la red neuronal

Arquitectura base: `TextVectorization → Embedding → [RNN | LSTM | GRU | BiLSTM] → Dropout → Dense(32) → Dropout → Dense(1, sigmoid)`.

Decisiones:
- **Pérdida:** `binary_crossentropy` (clasificación binaria con salida sigmoide).
- **Desbalance:** `class_weight` balanceado.
- **Regularización:** Dropout 0.4 + `EarlyStopping` (patience=3, restaura mejores pesos).
- **Comparación:** entrenamos 4 arquitecturas y elegimos la mejor por **F1 en validación** (nunca en test).

In [ ]:
EMBED_DIM, UNITS, DROPOUT = 64, 64, 0.4

def build_model(kind):
    inputs = keras.Input(shape=(), dtype=tf.string, name="texto")
    x = vectorizer(inputs)
    x = layers.Embedding(vectorizer.vocabulary_size(), EMBED_DIM, mask_zero=True)(x)
    if kind == "SimpleRNN":
        x = layers.SimpleRNN(UNITS)(x)
    elif kind == "LSTM":
        x = layers.LSTM(UNITS)(x)
    elif kind == "GRU":
        x = layers.GRU(UNITS)(x)
    elif kind == "BiLSTM":
        x = layers.Bidirectional(layers.LSTM(UNITS))(x)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(1, activation="sigmoid", name="prob_spam")(x)
    m = keras.Model(inputs, out, name=f"spam_{kind.lower()}")
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
              metrics=["accuracy", keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")])
    return m

build_model("BiLSTM").summary()

w = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_df["y"].values)
CLASS_WEIGHT = {0: float(w[0]), 1: float(w[1])}
print("class_weight:", CLASS_WEIGHT)

In [ ]:
os.makedirs("modelo", exist_ok=True)
modelos, historias, resumen = {}, {}, []

for nombre in ["SimpleRNN", "LSTM", "GRU", "BiLSTM"]:
    print(f"\n===== Entrenando {nombre} =====")
    tf.keras.utils.set_random_seed(SEED)
    m = build_model(nombre)
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
          keras.callbacks.ModelCheckpoint(f"modelo/ckpt_{nombre}.keras", monitor="val_loss", save_best_only=True)]
    h = m.fit(train_ds, validation_data=val_ds, epochs=25, class_weight=CLASS_WEIGHT, callbacks=cb, verbose=2)

    pv = m.predict(val_ds, verbose=0).ravel()
    yv = val_df["y"].values
    modelos[nombre], historias[nombre] = m, h
    resumen.append({"modelo": nombre, "épocas": len(h.history["loss"]),
                    "val_accuracy": accuracy_score(yv, pv >= .5), "val_precision": precision_score(yv, pv >= .5),
                    "val_recall": recall_score(yv, pv >= .5), "val_f1": f1_score(yv, pv >= .5),
                    "val_auc": roc_auc_score(yv, pv)})

tabla = pd.DataFrame(resumen).set_index("modelo").round(4)
tabla

## 7. Gráficas del entrenamiento

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey="row")
for j, (nombre, h) in enumerate(historias.items()):
    axes[0, j].plot(h.history["loss"], label="train"); axes[0, j].plot(h.history["val_loss"], label="val")
    axes[0, j].set_title(f"{nombre} — pérdida"); axes[0, j].set_xlabel("época")
    axes[1, j].plot(h.history["accuracy"], label="train"); axes[1, j].plot(h.history["val_accuracy"], label="val")
    axes[1, j].set_title(f"{nombre} — accuracy"); axes[1, j].set_xlabel("época")
axes[0, 0].legend(); axes[1, 0].legend()
plt.tight_layout(); plt.show()

mejor = tabla["val_f1"].idxmax()
print(f"✅ Mejor modelo según F1 en validación: {mejor}")

**Cómo leer las curvas:** si la pérdida de validación sube mientras la de entrenamiento baja, hay sobreajuste;
`EarlyStopping` detiene el entrenamiento y restaura los mejores pesos. Compara también la estabilidad entre
SimpleRNN (suele ser peor con secuencias largas por el desvanecimiento del gradiente) y LSTM/GRU.

## 8. Evaluación en el conjunto de prueba

In [ ]:
best_model = modelos[mejor]
y_test = test_df["y"].values
probs = best_model.predict(test_ds, verbose=0).ravel()
preds = (probs >= 0.5).astype(int)

metrics = {"modelo": mejor, "accuracy": accuracy_score(y_test, preds), "precision": precision_score(y_test, preds),
           "recall": recall_score(y_test, preds), "f1": f1_score(y_test, preds), "roc_auc": roc_auc_score(y_test, probs),
           "confusion_matrix": confusion_matrix(y_test, preds).tolist(), "n_test": int(len(y_test)), "threshold": 0.5}
print(classification_report(y_test, preds, target_names=["ham", "spam"], digits=4))
print(f"ROC-AUC: {metrics['roc_auc']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ConfusionMatrixDisplay.from_predictions(y_test, preds, display_labels=["ham", "spam"], cmap="Blues", ax=axes[0])
axes[0].set_title(f"Matriz de confusión — {mejor}")
RocCurveDisplay.from_predictions(y_test, probs, ax=axes[1]); axes[1].set_title("Curva ROC")
plt.tight_layout(); plt.show()

**Interpretación:** en detección de spam hay dos errores con costos distintos. Un **falso positivo** (ham marcado
como spam) puede ocultar un mensaje legítimo; un **falso negativo** deja pasar spam. Revisa precisión y recall
de la clase *spam* y decide el umbral según cuál error te importe más.

## 9. Predicciones de prueba (≥ 10)

In [ ]:
res = test_df.copy()
res["prob_spam"] = probs
res["predicción"] = np.where(preds == 1, "spam", "ham")
res["correcta"] = res["predicción"] == res["label"]

muestra = pd.concat([g.sample(6, random_state=SEED) for _, g in res.groupby("label")])
muestra = muestra[["text", "label", "predicción", "prob_spam", "correcta"]].reset_index(drop=True)
muestra["text"] = muestra["text"].str.slice(0, 80)
pd.set_option("display.max_colwidth", 90)
muestra.round(3)

In [ ]:
errores = res[~res["correcta"]][["text", "label", "predicción", "prob_spam"]].round(3)
print(f"Errores en test: {len(errores)} de {len(res)}")
errores.head(10)

## 10. Guardado del modelo

Guardamos el modelo en formato `.keras`. Incluye la capa `TextVectorization`, así que no hace falta guardar
un tokenizador aparte. Verificamos que al recargarlo produce las mismas probabilidades.

In [ ]:
os.makedirs("reports", exist_ok=True)
best_model.save("modelo/modelo_spam.keras")

config = {"model": mejor.lower(), "max_len": MAX_LEN, "vocab_size": VOCAB_SIZE,
          "vocab_real": int(vectorizer.vocabulary_size()), "embed_dim": EMBED_DIM, "units": UNITS,
          "dropout": DROPOUT, "batch_size": BATCH, "seed": SEED, "threshold": 0.5}
json.dump(config, open("modelo/config.json", "w"), indent=2)
json.dump(metrics, open("reports/metrics.json", "w"), indent=2)
tabla.to_csv("reports/comparacion_modelos.csv")
muestra.to_csv("reports/predicciones_test.csv", index=False)

# Verificación de recarga
reloaded = keras.models.load_model("modelo/modelo_spam.keras")
frases = ["WINNER!! You have won a £1000 prize. Call 09061701461 now to claim.",
          "Hey, are we still meeting at 6 for dinner?"]
p = reloaded.predict(tf.constant([clean_text(f) for f in frases]), verbose=0).ravel()
for f, pr in zip(frases, p):
    print(f"p(spam)={pr:.3f} → {'SPAM' if pr >= .5 else 'HAM'} | {f}")

p_orig = best_model.predict(test_ds, verbose=0).ravel()
p_reload = reloaded.predict(test_ds, verbose=0).ravel()
print("\nDiferencia máx. entre original y recargado:", float(np.abs(p_orig - p_reload).max()))

In [ ]:
# Descarga todo (modelo + reportes) como .zip
import shutil
for f in os.listdir("modelo"):
    if f.startswith("ckpt_"):
        os.remove(os.path.join("modelo", f))      # quita checkpoints intermedios
shutil.make_archive("resultados_proyecto", "zip", ".", "modelo")
shutil.make_archive("reportes_proyecto", "zip", ".", "reports")
try:
    from google.colab import files
    files.download("resultados_proyecto.zip"); files.download("reportes_proyecto.zip")
except ImportError:
    print("Fuera de Colab: los archivos están en ./modelo y ./reports")

## 11. Conclusiones

- Resume qué arquitectura ganó y por cuánto (tabla de la sección 6).
- Comenta el efecto de `class_weight` y del desbalance en precisión/recall.
- Analiza los errores de la sección 9: ¿qué tipo de mensajes confunde el modelo?
- Limitaciones: solo inglés, dataset pequeño. Mejoras: GloVe, ajuste de umbral, Transformers.

**Siguiente paso:** copia `modelo/` y `reports/` dentro de la carpeta del proyecto y ejecuta `streamlit run app.py`.